In [1]:
import polars as pl
import numpy as np
import time
from gensim.corpora import Dictionary
from gensim.models import LdaMulticore
from gensim.models import Phrases
from gensim.models.phrases import Phraser
from gensim.parsing.preprocessing import STOPWORDS
import joblib
from tqdm import tqdm
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
import os

/home/javclamar/Projects/tfg-sentiment-analysis/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
csv_reviews_with_categories = '../data/csv/yelp_reviews_with_business.csv'
yelp_stopwords = [
    "food", "good", "place", "service", "restaurant", "great", "time", 
    "go", "back", "really", "just", "like", "get", "one", "would", 
    "ive", "even", "also", "always", "got", "came", "went", "us", "im", "much"
]
stopwords_list = list(STOPWORDS) + [""] + yelp_stopwords

CATEGORIES = {
    "fast_food":   "Fast Food",
    "steakhouses": "Steakhouses",
}

def filter_by_category(input_csv: str, category_label: str) -> pl.DataFrame:
    """Loads only rows whose categories column contains the target label."""
    return (
        pl.scan_csv(input_csv)
        .filter(pl.col("categories").str.contains(category_label))
        .collect()
    )


def train_lda_for_category(input_csv: str, category_key: str, category_label: str):
    model_dir = f"../data/models/{category_key}/lda"
    os.makedirs(model_dir, exist_ok=True)

    df = filter_by_category(input_csv, category_label)
    print(f"[{category_label}] {len(df):,} reviews found")

    sample_size = min(300_000, len(df))
    df = df.sample(n=sample_size, seed=42)

    df = df.with_columns(
        pl.col("text")
        .str.replace_all(r"[^a-zA-Z\s]", "")
        .str.to_lowercase()
        .str.split(" ")
        .list.set_difference(stopwords_list) 
        .alias("tokens")
    )
    tokenized_docs = df["tokens"].to_list()
    del df
    
    bigram_detector = Phrases(tokenized_docs, min_count=10, threshold=20)
    
    bigram_model = Phraser(bigram_detector)
    
    tokenized_docs = [bigram_model[doc] for doc in tokenized_docs]

    dictionary = Dictionary(tokenized_docs)
    
    dictionary.filter_extremes(no_below=20, no_above=0.5, keep_n=20_000)

    corpus = [dictionary.doc2bow(t) for t in tokenized_docs]
    del tokenized_docs

    workers_count = max(1, os.cpu_count() - 1)
    lda = LdaMulticore(
        corpus=corpus,
        num_topics=10,
        id2word=dictionary,
        workers=workers_count,
        passes=3,
        chunksize=2_000,
        random_state=42,
    )

    lda.save(f"{model_dir}/lda_model.gensim")
    dictionary.save(f"{model_dir}/dictionary.dict")
    bigram_model.save(f"{model_dir}/bigram_model.pkl")
    
    print(f"[{category_label}] Se ha guardado el modelo LDA en {model_dir}")


def train_bertopic_for_category(input_csv: str, category_key: str, category_label: str):
    model_dir = f"../data/models/{category_key}/bertopic"
    os.makedirs(model_dir, exist_ok=True)

    model_file = f"{model_dir}/bertopic_model.pkl"
    
    df = filter_by_category(input_csv, category_label)
    print(f"[{category_label}] {len(df):,} reviews encontradas")
    sample_size = min(300_000, len(df))
    docs = df.sample(n=sample_size, seed=42)["text"].to_list()
    del df

    vectorizer_model = CountVectorizer(
        stop_words=stopwords_list, 
        ngram_range=(2, 3),
        min_df=10
    )

    topic_model = BERTopic(
        language="english", 
        calculate_probabilities=False, 
        verbose=True,
        vectorizer_model=vectorizer_model
    )    
    
    topic_model.fit_transform(docs)

    topic_model.save(
        model_file,
        serialization="pickle"
    )
    print(f"[{category_label}] Se ha guardado el modelo BERTOpic en {model_file}")


train_lda_for_category(csv_reviews_with_categories, "steakhouses", "Steakhouses")
train_bertopic_for_category(csv_reviews_with_categories, "fast_food", "Fast Food")
train_lda_for_category(csv_reviews_with_categories, "fast_food", "Fast Food")
train_bertopic_for_category(csv_reviews_with_categories, "steakhouses", "Steakhouses")

[Steakhouses] 240,040 reviews found
Detecting and applying bigrams...
[Steakhouses] LDA saved → ../data/models/steakhouses/lda
[Fast Food] 233,008 reviews found


2026-04-17 22:18:44,401 - BERTopic - Embedding - Transforming documents to embeddings.
Loading weights: 100%|█| 103/103 [00:00<00:00, 635.42it/s, Mate
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|█████████████| 7282/7282 [03:19<00:00, 36.54it/s]
2026-04-17 22:22:14,086 - BERTopic - Embedding - Completed ✓
2026-04-17 22:22:14,092 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-17 22:24:16,139 - BERTopic - Dimensionality - Completed ✓
2026-04-17 22:24:16,178 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-17 22:24:34,346 - BERTopic - Cluster - Completed ✓
2026-04-17 22:24:34,420 - BERTopic - Representation - Fine-tuning topics using representation 

[Fast Food] BERTopic saved → ../data/models/fast_food/bertopic/bertopic_model.pkl
[Fast Food] 233,008 reviews found
Detecting and applying bigrams...
[Fast Food] LDA saved → ../data/models/fast_food/lda
[Steakhouses] 240,040 reviews found


2026-04-17 22:28:29,667 - BERTopic - Embedding - Transforming documents to embeddings.
Loading weights: 100%|█| 103/103 [00:00<00:00, 722.18it/s, Mate
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|█████████████| 7502/7502 [04:18<00:00, 29.08it/s]
2026-04-17 22:32:59,758 - BERTopic - Embedding - Completed ✓
2026-04-17 22:32:59,760 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-17 22:34:47,076 - BERTopic - Dimensionality - Completed ✓
2026-04-17 22:34:47,085 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-17 22:35:10,994 - BERTopic - Cluster - Completed ✓
2026-04-17 22:35:11,040 - BERTopic - Representation - Fine-tuning topics using representation 

[Steakhouses] BERTopic saved → ../data/models/steakhouses/bertopic/bertopic_model.pkl


In [4]:
csv_reviews_with_categories = '../data/csv/yelp_reviews_with_business.csv'
csv_output_dir = "../results/topic_modeling/"
total_rows = 6_990_280
batch_size = 50000
expected_batches = total_rows // batch_size + (1 if total_rows % batch_size != 0 else 0)

CATEGORIES = {
    "fast_food":   "Fast Food",
    "steakhouses": "Steakhouses",
}

def load_category_models(category_key: str):
    """Load both models for a given category key."""
    lda = LdaMulticore.load(f"../data/models/{category_key}/lda/lda_model.gensim")
    dictionary = Dictionary.load(f"../data/models/{category_key}/lda/dictionary.dict")

    bigram_model = Phraser.load(f"../data/models/{category_key}/lda/bigram_model.pkl") 
    
    bertopic = BERTopic.load(f"../data/models/{category_key}/bertopic/bertopic_model.pkl")
    bertopic.verbose = False

    lda_topic_words = {
        topic_id: ", ".join(
            dictionary[wid] for wid, _ in lda.get_topic_terms(topic_id, topn=3)
        )
        for topic_id in range(lda.num_topics)
    }

    bert_labels = {-1: "outlier"}
    for topic_id in bertopic.get_topic_info()["Topic"]:
        if topic_id != -1:
            rep = bertopic.get_topic(topic_id)
            bert_labels[topic_id] = ", ".join(w for w, _ in rep[:3]) if rep else f"topic_{topic_id}"

    return lda, dictionary, bigram_model, bertopic, lda_topic_words, bert_labels


def infer_chunk(chunk: pl.DataFrame, lda, dictionary, bigram_model, bertopic, lda_topic_words, bert_labels):
    """Run LDA + BERTopic inference on a single Polars DataFrame chunk."""
    chunk_tokenized = chunk.with_columns(
        pl.col("text")
        .str.replace_all(r"[^a-zA-Z\s]", "")
        .str.to_lowercase()
        .str.split(" ")
        .list.set_difference(stopwords_list)
        .alias("tokens")
    )

    raw_tokens_list = chunk_tokenized["tokens"].to_list()
    bigramed_tokens = [bigram_model[doc] for doc in raw_tokens_list]

    bow_corpus = [dictionary.doc2bow(t) for t in bigramed_tokens]
    
    topic_distributions = lda[bow_corpus]

    dominant_topics, topic_probs = [], []
    for dist in topic_distributions:
        if dist:
            best_id, best_prob = max(dist, key=lambda x: x[1])
            dominant_topics.append(lda_topic_words[best_id])
            topic_probs.append(best_prob)
        else:
            dominant_topics.append("none")
            topic_probs.append(0.0)

    bert_topics, _ = bertopic.transform(chunk["text"].to_list())
    bert_labels_col = [bert_labels.get(t, f"topic_{t}") for t in bert_topics]

    return chunk.with_columns([
        pl.Series("lda_dominant_topic",   dominant_topics),
        pl.Series("lda_topic_probability", topic_probs),
        pl.Series("bertopic_topic",        bert_topics),
        pl.Series("bertopic_dominant_topic", bert_labels_col),
    ])

def topic_modeling_by_category(input_csv: str, out_dir: str):
    os.makedirs(out_dir, exist_ok=True)

    models = {
        key: load_category_models(key)
        for key in CATEGORIES
    }

    category_pattern = "|".join(CATEGORIES.values())

    start_time = time.time()
    reader = pl.read_csv_batched(input_csv, batch_size=batch_size)
    
    is_first_write = {key: True for key in CATEGORIES}

    with tqdm(total=expected_batches, desc="Category topic modeling", unit="batch") as pbar:
        while True:
            batches = reader.next_batches(1)
            if not batches:
                break

            chunk = batches[0]

            chunk = chunk.filter(
                pl.col("categories").str.contains(category_pattern)
            )

            if chunk.is_empty():
                pbar.update(1)
                continue

            category_expressions = [
                pl.when(pl.col("categories").str.contains(val))
                  .then(pl.lit(key))
                  .otherwise(None)
                for key, val in CATEGORIES.items()
            ]

            chunk = chunk.with_columns(
                pl.concat_list(category_expressions)
                  .list.drop_nulls()
                  .alias("category_key")
            ).explode("category_key")

            for key in CATEGORIES:
                sub = chunk.filter(pl.col("category_key") == key)
                if sub.is_empty():
                    continue
                
                lda, dictionary, bigram_model, bertopic, lda_topic_words, bert_labels = models[key]
                processed_sub = infer_chunk(sub, lda, dictionary, bigram_model, bertopic, lda_topic_words, bert_labels)

                output_csv = os.path.join(out_dir, f"yelp_topics_{key}.csv")
                
                mode = "wb" if is_first_write[key] else "ab"
                
                with open(output_csv, mode) as f:
                    processed_sub.write_csv(f, include_header=is_first_write[key])
                
                is_first_write[key] = False

            pbar.update(1)

    print(f"Done in {(time.time() - start_time) / 60:.2f} min. Files saved to: {out_dir}")

# Execute the function
topic_modeling_by_category(csv_reviews_with_categories, csv_output_dir)

Category topic modeling:   0%|                                                                                      | 0/140 [00:00<?, ?batch/s]2026-04-17 23:42:53,070 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-04-17 23:42:53,871 - BERTopic - Dimensionality - Completed ✓
2026-04-17 23:42:53,872 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-04-17 23:42:54,152 - BERTopic - Cluster - Completed ✓
2026-04-17 23:42:56,188 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-04-17 23:42:57,163 - BERTopic - Dimensionality - Completed ✓
2026-04-17 23:42:57,164 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-04-17 23:42:57,529 - BERTopic - Cluster - Completed ✓
Category topic modeling:   1%|▌                                                                             | 1/140 [00:06<16:07,  6.96s/batch]2026-04-17 23:42:59,447 - BERTopic - Dimensionality - Reducing dimensional

KeyboardInterrupt: 